In [11]:
import pandas as pd
import os

# Set the directory path
Ev_raw_dir = r"C:\Users\jiahan2\Desktop\ACE_592\Final Project\Raw_Data"

# Load the CSV file
file_path = os.path.join(Ev_raw_dir, "evwatts.public.session.csv")
EV_session_raw_df = pd.read_csv(file_path)

EV_session_raw_df

,session_id,evse_id,connector_id,start_datetime,end_datetime,total_duration,charge_duration,energy_kwh,start_soc,end_soc,flag_id
0,11562,5,5,2020-03-01 09:37:44,2020-03-01 11:14:59,1.621389,1.615556,6.170000,NaN,NaN,0
1,11563,116,116,2020-03-01 10:02:24,2020-03-01 11:08:14,1.096944,1.089722,5.212000,NaN,NaN,0
2,11564,72,72,2020-03-01 10:06:08,2020-03-01 11:08:10,1.033611,1.024722,3.392000,NaN,NaN,0
3,11565,1,1,2020-03-01 10:01:12,2020-03-01 11:07:09,1.099722,1.076111,6.521000,NaN,NaN,0
4,11566,0,138,2020-02-29 19:09:03,2020-03-01 10:47:16,15.636667,12.658889,80.527000,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...
13937230,14715979,27099,36881,2022-12-28 19:51:30,2022-12-29 00:30:53,4.656389,4.655556,45.023398,NaN,NaN,0
13937231,14715980,27107,36889,2022-10-17 09:32:10,2022-10-17 09:37:22,0.086667,0.710000,8.217523,NaN,NaN,0
13937232,14715981,27107,36889,2022-10-22 16:38:44,2022-10-22 22:21:32,5.713333,3.056389,10.849586,NaN,NaN,0
13937233,14715982,27107,36889,2022-10-29 15:29:15,2022-10-29 15:54:17,0.417222,0.416667,2.743180,NaN,NaN,0


In [12]:
#Read the evse file that contain data on evse's metro_area, region...
file_path2 = os.path.join(Ev_raw_dir, "evwatts.public.evse.csv")
EV_evse_raw_df = pd.read_csv(file_path2)

EV_evse_raw_df

,evse_id,metro_area,land_use,region,num_ports,charge_level,venue,pricing
0,6034,Undesignated,Metro Area,Mountain,1,DCFC,Corridor,Paid
1,6065,Undesignated,Metro Area,Mountain,1,DCFC,Corridor,Paid
2,6100,"Phoenix-Mesa-Chandler, AZ Metro Area",Metro Area,Mountain,1,DCFC,Undesignated,Free
3,6155,Undesignated,Metro Area,Mountain,1,DCFC,Corridor,Paid
4,1315,"Denver-Aurora-Lakewood, CO Metro Area",Metro Area,Mountain,1,DCFC,Corridor,Free
...,...,...,...,...,...,...,...,...
36921,17417,Undesignated,Metro Area,West South Central,2,L2,Transit Facility,Paid
36922,17602,Undesignated,Metro Area,West South Central,2,L2,Transit Facility,Paid
36923,17562,Undesignated,Metro Area,West South Central,2,L2,Transit Facility,Paid
36924,23604,Undesignated,Metro Area,West South Central,2,L2,Transit Facility,Paid


In [13]:
#Check the overlapped id between evse and session based on evse_id
session_ids = set(EV_session_raw_df['evse_id'].unique())
evse_ids    = set(EV_evse_raw_df['evse_id'].unique())

print(f"\nUnique evse_id in Session data : {len(session_ids)}")
print(f"Unique evse_id in EVSE data    : {len(evse_ids)}")
print(f"Matching evse_ids              : {len(session_ids & evse_ids)}")
print(f"In Session but NOT in EVSE     : {len(session_ids - evse_ids)}")
print(f"In EVSE but NOT in Session     : {len(evse_ids - session_ids)}")

# 95% evse_id are matched; dropped the unmatched ids, and merge based on matched evse_id



Unique evse_id in Session data : 38458
Unique evse_id in EVSE data    : 36926
Matching evse_ids              : 36894
In Session but NOT in EVSE     : 1564
In EVSE but NOT in Session     : 32


In [14]:
# ── Keep Only Matched Sessions (Inner Join) ────────────────────────────────────
EV_merged_df = EV_session_raw_df.merge(EV_evse_raw_df, on='evse_id', how='inner')

EV_merged_df.columns

Index(['session_id', 'evse_id', 'connector_id', 'start_datetime',
       'end_datetime', 'total_duration', 'charge_duration', 'energy_kwh',
       'start_soc', 'end_soc', 'flag_id', 'metro_area', 'land_use', 'region',
       'num_ports', 'charge_level', 'venue', 'pricing'],
      dtype='str')

In [15]:
EV_merged_df

,session_id,evse_id,connector_id,start_datetime,end_datetime,total_duration,charge_duration,energy_kwh,start_soc,end_soc,flag_id,metro_area,land_use,region,num_ports,charge_level,venue,pricing
0,11562,5,5,2020-03-01 09:37:44,2020-03-01 11:14:59,1.621389,1.615556,6.170000,NaN,NaN,0,Undesignated,Undesignated,Middle Atlantic,2,L2,Municipal Building,Undesignated
1,11563,116,116,2020-03-01 10:02:24,2020-03-01 11:08:14,1.096944,1.089722,5.212000,NaN,NaN,0,Undesignated,Metro Area,Middle Atlantic,2,L2,Medical or Educational Campus,Undesignated
2,11564,72,72,2020-03-01 10:06:08,2020-03-01 11:08:10,1.033611,1.024722,3.392000,NaN,NaN,0,"Rochester, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated
3,11565,1,1,2020-03-01 10:01:12,2020-03-01 11:07:09,1.099722,1.076111,6.521000,NaN,NaN,0,"Albany-Schenectady-Troy, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated
4,11566,0,138,2020-02-29 19:09:03,2020-03-01 10:47:16,15.636667,12.658889,80.527000,NaN,NaN,0,"Albany-Schenectady-Troy, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13491249,14715979,27099,36881,2022-12-28 19:51:30,2022-12-29 00:30:53,4.656389,4.655556,45.023398,NaN,NaN,0,Undesignated,Metro Area,West South Central,2,L2,Multi-use Parking Garage/Lot,Undesignated
13491250,14715980,27107,36889,2022-10-17 09:32:10,2022-10-17 09:37:22,0.086667,0.710000,8.217523,NaN,NaN,0,Undesignated,Metro Area,West South Central,2,L2,Multi-use Parking Garage/Lot,Undesignated
13491251,14715981,27107,36889,2022-10-22 16:38:44,2022-10-22 22:21:32,5.713333,3.056389,10.849586,NaN,NaN,0,Undesignated,Metro Area,West South Central,2,L2,Multi-use Parking Garage/Lot,Undesignated
13491252,14715982,27107,36889,2022-10-29 15:29:15,2022-10-29 15:54:17,0.417222,0.416667,2.743180,NaN,NaN,0,Undesignated,Metro Area,West South Central,2,L2,Multi-use Parking Garage/Lot,Undesignated


In [16]:
# ── Check metro_area Undesignated ─────────────────────────────────────────────
undesignated_count = (EV_merged_df['metro_area'] == 'Undesignated').sum()
total_count = len(EV_merged_df)

print(f"Undesignated metro_area count : {undesignated_count}")
print(f"Total records                 : {total_count}")
print(f"Undesignated %                : {(undesignated_count/total_count)*100:.2f}%")

# ── Full breakdown of all metro_area values ────────────────────────────────────
print("\nAll metro_area value counts:")
print(EV_merged_df['metro_area'].value_counts())


Undesignated metro_area count : 4780102
Total records                 : 13491254
Undesignated %                : 35.43%

All metro_area value counts:
metro_area
Undesignated                                               4780102
Portland-Vancouver-Hillsboro, OR-WA Metro Area              818951
Washington-Arlington-Alexandria, DC-VA-MD-WV Metro Area     706192
Baltimore-Columbia-Towson, MD Metro Area                    614239
Detroit-Warren-Dearborn, MI Metro Area                      559108
Las Vegas-Henderson-Paradise, NV Metro Area                 487831
Philadelphia-Camden-Wilmington, PA-NJ-DE-MD Metro Area      466245
New York-Newark-Jersey City, NY-NJ-PA Metro Area            405086
Boston-Cambridge-Newton, MA-NH Metro Area                   367295
Burlington-South Burlington, VT Metro Area                  365514
Los Angeles-Long Beach-Anaheim, CA Metro Area               360303
Kansas City, MO-KS Metro Area                               321813
Phoenix-Mesa-Chandler, AZ Metro Are

In [17]:
# ── Drop Undesignated metro_area ───────────────────────────────────────────────
EV_merged_df = EV_merged_df[EV_merged_df['metro_area'] != 'Undesignated']

print("Shape after dropping Undesignated:", EV_merged_df.shape)
print("\nRemaining metro_area values:")
print(EV_merged_df['metro_area'].value_counts())

Shape after dropping Undesignated: (8711152, 18)

Remaining metro_area values:
metro_area
Portland-Vancouver-Hillsboro, OR-WA Metro Area             818951
Washington-Arlington-Alexandria, DC-VA-MD-WV Metro Area    706192
Baltimore-Columbia-Towson, MD Metro Area                   614239
Detroit-Warren-Dearborn, MI Metro Area                     559108
Las Vegas-Henderson-Paradise, NV Metro Area                487831
Philadelphia-Camden-Wilmington, PA-NJ-DE-MD Metro Area     466245
New York-Newark-Jersey City, NY-NJ-PA Metro Area           405086
Boston-Cambridge-Newton, MA-NH Metro Area                  367295
Burlington-South Burlington, VT Metro Area                 365514
Los Angeles-Long Beach-Anaheim, CA Metro Area              360303
Kansas City, MO-KS Metro Area                              321813
Phoenix-Mesa-Chandler, AZ Metro Area                       303602
Providence-Warwick, RI-MA Metro Area                       302610
Denver-Aurora-Lakewood, CO Metro Area               

In [18]:
# ── Check flag_id values ───────────────────────────────────────────────────────
#   0	no flag
# 	1	unrealistic kW (Note: flag on vehicle_session only includes sessions longer than 1 min)
#	2	Multiple overlapping (concurrent) sessions at port or multiple overlapping (concurrent) vehicle trips
#	4	start_soc > end_soc
#	8	kWh consumption greater than battery kWh  (120%)
#	16	duplicate row
#	32	charging kWh greater than battery kWh (120%)
#	64	insignificant session length (less than 2.5 min)
#	128	Sessions with 0 kWh transferred
#	256	Low kWh Session ( <0.3 kWh transferred)
#	512	Vehicle trip opverlaps Vehicle Charging Session
#	1024	Energetics Internal Usage
#	2048	Vehicle Trips with high average Wh/mi (Thresholds: SUV > 1100 Wh/mi, Sedan > 650 Wh/mi, Minivan > 1100 Wh/mi)
#	4096 	Estimated Home Location (Telematics)
#	8192	Energy kWh is Null
#	16384	EVSEs that do not observe daylight savings time
#	32768   End Datetime missing when Total Duration and Start Datetime is present
#	65536   Negative Charge Duration

print("flag_id value counts:")
print(EV_merged_df['flag_id'].value_counts())

print(f"\nUnique flag_id values: {EV_merged_df['flag_id'].unique()}")
print(f"Missing values       : {EV_merged_df['flag_id'].isnull().sum()}")

flag_id value counts:
flag_id
0        8125327
32        169024
256       129853
64         96766
320        85425
2          49499
4          41002
192         5871
128         3815
34          1342
258          578
36           551
322          507
324          463
260          391
68           231
1            150
66            87
65            73
32768         56
130           26
257           20
196           16
66560         14
5             12
132           11
194            8
6              7
321            7
69             5
3              4
33             3
97             2
96             2
33024          2
101            1
32770          1
Name: count, dtype: int64

Unique flag_id values: [    0    64   320   192   256   128     2   258     4   324    66    68
   130   260    32     6    36   194     5    65    69     1    97   322
    34     3   132   196    33   257    96   101   321 66560 32768 32770
 33024]
Missing values       : 0


In [19]:
# ── Drop observations with flag_id > 0 ────────────────────────────────────────
EV_merged_df = EV_merged_df[EV_merged_df['flag_id'] == 0]

# ── Validate ───────────────────────────────────────────────────────────────────
print("Shape after dropping flag_id > 0:", EV_merged_df.shape)
print("\nRemaining flag_id values:")
print(EV_merged_df['flag_id'].value_counts())

Shape after dropping flag_id > 0: (8125327, 18)

Remaining flag_id values:
flag_id
0    8125327
Name: count, dtype: int64


In [24]:
# Investigate Abnormal Energy Consumption Values 

# Zero energy sessions
print("Zero kWh Sessions:")
print(f"  Count : {(EV_merged_df['energy_kwh'] == 0).sum():,}")

# Sessions > 150 kWh broken down by charge level
print("\nSessions > 150 kWh by Charge Level:")
print(EV_merged_df[EV_merged_df['energy_kwh'] > 150]['charge_level'].value_counts())

# Sessions > 150 kWh broken down by venue
print("\nSessions > 150 kWh by Venue:")
print(EV_merged_df[EV_merged_df['energy_kwh'] > 150]['venue'].value_counts().head(10))

# Look at extreme sessions
print("\nTop 10 Highest Energy Sessions:")
print(EV_merged_df.nlargest(10, 'energy_kwh')[
    ['session_id', 'evse_id', 'start_datetime', 'end_datetime', 
     'charge_duration', 'energy_kwh', 'charge_level', 'metro_area']
])

Zero kWh Sessions:
  Count : 13,397

Sessions > 150 kWh by Charge Level:
charge_level
DCFC    26078
L2         43
Name: count, dtype: int64

Sessions > 150 kWh by Venue:
venue
Undesignated                     24641
Corridor                          1437
Fleet                               18
Single Family Residential           11
Multi-Unit Dwelling                  5
Multi-use Parking Garage/Lot         3
Transit Facility                     2
Business Office                      2
Medical or Educational Campus        1
Municipal Building                   1
Name: count, dtype: int64

Top 10 Highest Energy Sessions:
         session_id  evse_id      start_datetime         end_datetime  \
1232793     1255915     1081 2021-02-19 23:37:26  2021-02-21 15:17:40   
286405       298884     1849 2020-03-13 11:31:56  2020-03-19 20:27:17   
4514479     5051181     1862 2021-07-03 16:48:16  2021-07-19 00:50:40   
942892       959172     1849 2020-12-24 10:38:36  2020-12-28 05:16:22   
6853224   

In [29]:
# 50–350 kW DC Fast Chargers (DCFC) are common; Keep these but worth capping extremes
# Drop Observations with Abnormal Energy Consumption 

# 1. Drop zero kWh sessions (failed/ghost sessions)
before = len(EV_merged_df)
EV_merged_df = EV_merged_df[EV_merged_df['energy_kwh'] > 0]
print(f"Dropped zero kWh     : {before - len(EV_merged_df):,} rows")

# 2. Drop L2 sessions > 150 kWh (physically implausible for L2)
before = len(EV_merged_df)
EV_merged_df = EV_merged_df[~((EV_merged_df['charge_level'] == 'L2') & 
                               (EV_merged_df['energy_kwh'] > 150))]
print(f"Dropped L2 > 150 kWh : {before - len(EV_merged_df):,} rows")

# 3. Cap DCFC at 99.9th percentile within DCFC sessions only
dcfc_cap = EV_merged_df[EV_merged_df['charge_level'] == 'DCFC']['energy_kwh'].quantile(0.999)
print(f"\nDCFC 99.9th pct cap  : {dcfc_cap:.2f} kWh")

before = len(EV_merged_df)
EV_merged_df = EV_merged_df[~((EV_merged_df['charge_level'] == 'DCFC') & 
                               (EV_merged_df['energy_kwh'] > dcfc_cap))]
print(f"Dropped DCFC outliers: {before - len(EV_merged_df):,} rows")

print(f"\nDataset Shape  : {EV_merged_df.shape}")
print(f"energy_kwh Min       : {EV_merged_df['energy_kwh'].min():.2f}")
print(f"energy_kwh Max       : {EV_merged_df['energy_kwh'].max():.2f}")
print(f"energy_kwh Mean      : {EV_merged_df['energy_kwh'].mean():.2f}")
print(f"energy_kwh Median    : {EV_merged_df['energy_kwh'].median():.2f}")

Dropped zero kWh     : 0 rows
Dropped L2 > 150 kWh : 0 rows

DCFC 99.9th pct cap  : 327.65 kWh
Dropped DCFC outliers: 1,024 rows

Dataset Shape  : (8109838, 21)
energy_kwh Min       : 0.30
energy_kwh Max       : 327.64
energy_kwh Mean      : 16.67
energy_kwh Median    : 10.85


In [33]:
EV_merged_df['year']  = EV_merged_df['start_datetime'].dt.year
EV_merged_df['month'] = EV_merged_df['start_datetime'].dt.month

In [34]:
EV_merged_df

,session_id,evse_id,connector_id,start_datetime,end_datetime,total_duration,charge_duration,energy_kwh,start_soc,end_soc,...,metro_area,land_use,region,num_ports,charge_level,venue,pricing,date,year,month
2,11564,72,72,2020-03-01 10:06:08,2020-03-01 11:08:10,1.033611,1.024722,3.392000,NaN,NaN,...,"Rochester, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated,2020-03-01,2020,3
3,11565,1,1,2020-03-01 10:01:12,2020-03-01 11:07:09,1.099722,1.076111,6.521000,NaN,NaN,...,"Albany-Schenectady-Troy, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated,2020-03-01,2020,3
4,11566,0,138,2020-02-29 19:09:03,2020-03-01 10:47:16,15.636667,12.658889,80.527000,NaN,NaN,...,"Albany-Schenectady-Troy, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated,2020-02-29,2020,2
7,11569,72,201,2020-03-01 08:43:22,2020-03-01 10:09:41,1.438611,1.416389,8.949000,NaN,NaN,...,"Rochester, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated,2020-03-01,2020,3
8,11570,14,14,2020-03-01 08:29:45,2020-03-01 09:56:48,1.451667,1.446667,4.238000,NaN,NaN,...,"Rochester, NY Metro Area",Metro Area,Middle Atlantic,2,L2,Municipal Building,Undesignated,2020-03-01,2020,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13491078,14715813,37982,50241,2022-11-05 13:13:50,2022-11-05 15:51:54,2.634444,2.106944,12.146016,NaN,NaN,...,"Washington-Arlington-Alexandria, DC-VA-MD-WV M...",Metro Area,South Atlantic,1,L2,Hotel,Undesignated,2022-11-05,2022,11
13491079,14715814,37982,50241,2022-12-31 17:53:27,2022-12-31 18:55:38,1.036389,1.033611,3.863187,NaN,NaN,...,"Washington-Arlington-Alexandria, DC-VA-MD-WV M...",Metro Area,South Atlantic,1,L2,Hotel,Undesignated,2022-12-31,2022,12
13491182,14837027,5819,9885,2022-12-31 08:56:49,2023-01-01 03:00:35,18.062778,18.063889,54.270000,NaN,NaN,...,"Dallas-Fort Worth-Arlington, TX Metro Area",Metro Area,West South Central,1,L2,Transit Facility,Free,2022-12-31,2022,12
13491183,14837028,5821,9887,2022-12-31 05:10:34,2022-12-31 08:10:34,3.000000,54.669444,18.650000,NaN,NaN,...,"Dallas-Fort Worth-Arlington, TX Metro Area",Metro Area,West South Central,1,L2,Transit Facility,Free,2022-12-31,2022,12


In [27]:
# ── Extract Date from start_datetime ──────────────────────────────────────────
EV_merged_df['start_datetime'] = pd.to_datetime(EV_merged_df['start_datetime'])

EV_merged_df['date'] = EV_merged_df['start_datetime'].dt.date

print("Date Range:")
print("  Start:", EV_merged_df['date'].min())
print("  End:  ", EV_merged_df['date'].max())

Date Range:
  Start: 2019-06-28
  End:   2022-12-31


In [30]:
# ── Count and List All Metro Areas ────────────────────────────────────────────
metro_counts = EV_merged_df['metro_area'].value_counts().reset_index()
metro_counts.columns = ['metro_area', 'session_count']

print(f"Total Unique Metro Areas: {EV_merged_df['metro_area'].nunique()}")
print(f"\n{'#':<5} {'Metro Area':<60} {'Sessions':>10}")
print("-" * 78)
for i, row in metro_counts.iterrows():
    print(f"{i+1:<5} {row['metro_area']:<60} {row['session_count']:>10,}")

Total Unique Metro Areas: 29

#     Metro Area                                                     Sessions
------------------------------------------------------------------------------
1     Portland-Vancouver-Hillsboro, OR-WA Metro Area                  773,378
2     Washington-Arlington-Alexandria, DC-VA-MD-WV Metro Area         658,753
3     Baltimore-Columbia-Towson, MD Metro Area                        574,911
4     Detroit-Warren-Dearborn, MI Metro Area                          512,675
5     Las Vegas-Henderson-Paradise, NV Metro Area                     454,181
6     Philadelphia-Camden-Wilmington, PA-NJ-DE-MD Metro Area          439,601
7     New York-Newark-Jersey City, NY-NJ-PA Metro Area                365,660
8     Burlington-South Burlington, VT Metro Area                      344,326
9     Boston-Cambridge-Newton, MA-NH Metro Area                       341,657
10    Los Angeles-Long Beach-Anaheim, CA Metro Area                   338,892
11    Kansas City, MO-KS Metro Ar

In [41]:
import os
import pyarrow as pa
import pyarrow.parquet as pq

# ── Check for problematic dtypes ───────────────────────────────────────────────
print("Current dtypes:")
print(EV_merged_df.dtypes)

# ── Fix: Convert ALL non-standard dtypes to basic types ───────────────────────
EV_merged_clean = EV_merged_df.copy()

for col in EV_merged_clean.columns:
    if EV_merged_clean[col].dtype == 'object':
        EV_merged_clean[col] = EV_merged_clean[col].astype(str)
    elif pd.api.types.is_datetime64_any_dtype(EV_merged_clean[col]):
        EV_merged_clean[col] = EV_merged_clean[col].astype(str)
    elif pd.api.types.is_period_dtype(EV_merged_clean[col]):
        EV_merged_clean[col] = EV_merged_clean[col].astype(str)

print("\nCleaned dtypes:")
print(EV_merged_clean.dtypes)

# ── Save using PyArrow directly (bypass pandas) ───────────────────────────────
output_path = os.path.join(Ev_raw_dir, "EV_merged.parquet")

table = pa.Table.from_pandas(EV_merged_clean, preserve_index=False)
pq.write_table(table, output_path)

print(f"\nSaved : {output_path}")
print(f"Shape : {EV_merged_clean.shape}")

# ── Verify ─────────────────────────────────────────────────────────────────────
verify_df = pd.read_parquet(output_path)
print(f"\nVerification - Shape : {verify_df.shape}")
print(f"Verification - Cols  : {verify_df.columns.tolist()}")

Current dtypes:
session_id                  int64
evse_id                     int64
connector_id                int64
start_datetime     datetime64[us]
end_datetime                  str
total_duration            float64
charge_duration           float64
energy_kwh                float64
start_soc                 float64
end_soc                   float64
flag_id                     int64
metro_area                    str
land_use                      str
region                        str
num_ports                   int64
charge_level                  str
venue                         str
pricing                       str
date                       object
year                        int32
month                       int32
dtype: object


C:\Users\jiahan2\AppData\Local\Temp\ipykernel_24720\3783452897.py:17: Pandas4Warning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types.is_period_dtype(EV_merged_clean[col]):
C:\Users\jiahan2\AppData\Local\Temp\ipykernel_24720\3783452897.py:17: Pandas4Warning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types.is_period_dtype(EV_merged_clean[col]):
C:\Users\jiahan2\AppData\Local\Temp\ipykernel_24720\3783452897.py:17: Pandas4Warning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  elif pd.api.types.is_period_dtype(EV_merged_clean[col]):



Cleaned dtypes:
session_id           int64
evse_id              int64
connector_id         int64
start_datetime         str
end_datetime           str
total_duration     float64
charge_duration    float64
energy_kwh         float64
start_soc          float64
end_soc            float64
flag_id              int64
metro_area             str
land_use               str
region                 str
num_ports            int64
charge_level           str
venue                  str
pricing                str
date                   str
year                 int32
month                int32
dtype: object

Saved : C:\Users\jiahan2\Desktop\ACE_592\Final Project\Raw_Data\EV_merged.parquet
Shape : (8109838, 21)


ArrowKeyError: A type extension with name pandas.period already defined